In [ ]:
%%writefile problem1.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__
void compute(int a , int b , int c , int *result){
  *result = (a * b) - c ;
}

int main(){
  int result ;
  int *d_result ;

  cudaMalloc(&d_result , sizeof(int));
  compute<<<1,1>>>(10 , 4 , 7 ,d_result);
  cudaMemcpy(&result , d_result , sizeof(int),cudaMemcpyDeviceToHost);
  printf("(10 * 4 ) - 7 = %d\n",result);
  cudaFree(d_result);
  return 0 ;
}

In [ ]:
!nvcc problem1.cu -o problem1 && ./problem1

In [ ]:
%%writefile problem2.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void square(int *arr){
  int i = threadIdx.x;
  arr[i] = arr[i] * 2;
}

int main(){
  int arr[6] ={1,2,3,4,5,6};
  int *d_arr ;

  cudaMalloc(&d_arr , 6*sizeof(int));
  cudaMemcpy(d_arr , arr , 6* sizeof(int),cudaMemcpyHostToDevice);
  square<<<1,6>>>(d_arr);
  cudaMemcpy(arr , d_arr , 6*sizeof(int),cudaMemcpyDeviceToHost);

  for (int i = 0 ; i < 6 ; i++){
    printf("%d\n",arr[i]);
  }
  return 0 ;
}

In [ ]:
!nvcc problem2.cu  -o problem2 && ./problem2

In [ ]:
%%writefile problem3.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void increment(int *arr){
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  arr[i] += 10;
}

int main(){
  int arr[8] = {1,2,3,4,5,6,7,8};
  int *d_arr;

  cudaMalloc(&d_arr , 8*sizeof(int));
  cudaMemcpy(d_arr , arr , 8*sizeof(int) , cudaMemcpyHostToDevice);
  increment<<<2 , 4>>>(d_arr);
  cudaMemcpy(arr , d_arr , 8*sizeof(int) , cudaMemcpyDeviceToHost);

  for (int i = 0 ; i < 8 ; i++){
    printf("%d\n",arr[i]);
  }

  cudaDeviceSynchronize();
  cudaFree(d_arr);
  return 0 ;
}

In [ ]:
!nvcc problem3.cu -o problem3 && ./problem3

In [ ]:
%%writefile problem4.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void process (int *arr , int n){
  int i = blockIdx.x * blockDim.x + threadIdx.x;

  if(i < n){
    arr[i] *= 3;
  }
}

int main(){
  int arr[10] = {1,2,3,4,5,6,7,8,9,10};
  int *device_arr ;
  int n = sizeof(arr) / sizeof(arr[0]);

  cudaMalloc(&device_arr , 10*sizeof(int));
  cudaMemcpy(device_arr , arr , 10*sizeof(int) , cudaMemcpyHostToDevice);
  process<<<4 , 4>>>(device_arr ,n);
  cudaMemcpy(arr , device_arr , 10*sizeof(int) , cudaMemcpyDeviceToHost);

   for (int i = 0 ; i < n ; i++){
    printf("%d\n",arr[i]);
  }

  return 0 ;
}

In [ ]:
!nvcc problem4.cu -o problem4 && ./problem4

In [ ]:
%%writefile problem5.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void process (int *arr , int n){
  __shared__ int title[4];

  int i = blockIdx.x * blockDim.x + threadIdx.x;

  if (i < n){
    title[threadIdx.x] = arr[i];
  }

  __syncthreads();
  int sum = 0 ;
  if (threadIdx.x == 0){
    for (int j = 0 ; j < blockDim.x ; j++){
      sum += title[j];
    }
  }
  title[0] = sum ;

  __syncthreads();

  if (i < n){
    arr[i] = title[0];
  }
}


int main(){
  int arr[8] = {1,2,3,4,5,6,7,8};
  int *d_arr ; 
  int n = 8;

  cudaMalloc(&d_arr , n*sizeof(int));
  cudaMemcpy(d_arr , arr , n * sizeof(int) , cudaMemcpyHostToDevice);
  process<<<2 ,4>>>(d_arr , n);
  cudaMemcpy(arr , d_arr , n * sizeof(int) , cudaMemcpyDeviceToHost);
  
   for (int i = 0 ; i < n ; i++){
    printf("%d\n",arr[i]);
  }
  cudaFree(d_arr);
  return 0 ;
}

In [ ]:
!nvcc problem5.cu -o problem5 && ./problem5